In [1]:
# Parameters
run_id = "1cc41367-e95c-4686-b4a7-98f690865869"
artifacts_dir = "/home/adnoman/projects/aml_gan/AMLend2end/artifacts/runs/1cc41367-e95c-4686-b4a7-98f690865869"
sample_size = None
epochs = None
threshold = None


# Create node embeddings feature groups.

Up until now we use feature engineering, feature store and model training to create node embedding. We will now materialise this as node embeddings feature group. This feature group will be used to train anomaly detection model.

![Feature Stores](./images/online_offline_fs.png)

---
**NOTE**: 

In real life scenarios financial transaction are dynamically evolving graphs. If live Transaction Monitoring System is based on graph or node embeddings then this will require 1st to update the graph and node representations after new transactions arrive. Recomputing entire graph for every newly arrived transaction will lead to unaxeptable delayes and even monitoring system failures. This problem  will be more sever if large amount of updates happen in a short time window.

Contact us at Logical Clocks and we will help you to setup end to end graph based deep anomaly detection live Transaction Monitoring Systems. 

---

## Query Model Repository for best node embeddings model

In [2]:
# Setup for local execution
import os
import json
import pandas as pd
import numpy as np

# Define paths
BASE_PATH = os.path.dirname(os.path.abspath("__file__"))
TRAINING_DATA_PATH = os.path.join(BASE_PATH, "training_data")
OUTPUT_PATH = os.path.join(BASE_PATH, "output")
MODELS_PATH = os.path.join(BASE_PATH, "models")
RESOURCES_PATH = os.path.join(BASE_PATH, "Resources")

print(f"Training data: {TRAINING_DATA_PATH}")
print(f"Output: {OUTPUT_PATH}")

Training data: /home/adnoman/projects/aml_gan/AMLend2end/training_data
Output: /home/adnoman/projects/aml_gan/AMLend2end/output


In [3]:
# Find the latest model directory
model_dirs = [d for d in os.listdir(MODELS_PATH) if d.startswith('node_embeddings_')]
if model_dirs:
    latest_model_dir = os.path.join(MODELS_PATH, sorted(model_dirs)[-1])
    print(f"Found model: {latest_model_dir}")
    
    # Load metadata
    with open(os.path.join(latest_model_dir, 'metadata.json'), 'r') as f:
        metadata = json.load(f)
    print(f"Model metrics: {metadata['metrics']}")
    print(f"Hyperparameters: {metadata['hyperparameters']}")
else:
    print("No model found! Run notebook 4 first.")

Found model: /home/adnoman/projects/aml_gan/AMLend2end/models/node_embeddings_e2366de4
Model metrics: {'accuracy': 0.8891156462585034}
Hyperparameters: {'walk_number': 2, 'walk_length': 2, 'emb_size': 32}


In [4]:
# Load node embeddings from notebook 4
embeddings_path = os.path.join(TRAINING_DATA_PATH, "node_embeddings.csv")
node_embeddings_df = pd.read_csv(embeddings_path)

print(f"Loaded embeddings shape: {node_embeddings_df.shape}")
node_embeddings_df.head()

Loaded embeddings shape: (7347, 33)


,node_id,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,emb_7,emb_8,...,emb_22,emb_23,emb_24,emb_25,emb_26,emb_27,emb_28,emb_29,emb_30,emb_31
0,3aa9646b,0.016653,-0.022373,0.029435,0.024201,0.008518,0.019205,0.000119,-0.004495,-0.004817,...,0.023774,-0.021663,-0.004356,0.005559,-0.011248,0.003737,-0.005930,-0.003396,0.022684,-0.015906
1,1e46e726,0.010128,0.018849,-0.023076,-0.001208,-0.010285,-0.014407,0.001506,-0.014259,0.013675,...,0.020794,-0.017354,-0.014928,0.011635,-0.017038,-0.027755,-0.001501,-0.024140,-0.030260,0.026108
2,49203bc3,-0.002191,-0.026739,0.005370,-0.004439,0.000192,0.016502,-0.002795,-0.011158,-0.013902,...,0.018414,0.012780,-0.029606,-0.030296,0.002708,0.010662,0.014418,-0.027512,0.006895,-0.027158
3,a74d1101,-0.003392,-0.017784,-0.009782,-0.026463,-0.020670,0.011362,-0.030823,-0.024861,0.010247,...,0.015851,0.028176,-0.006826,0.002322,-0.015091,-0.021084,-0.028587,-0.004787,0.010662,-0.022505
4,616d4505,0.022685,-0.000191,-0.003965,-0.002960,-0.015268,-0.004893,0.023440,0.018718,-0.019463,...,-0.026782,-0.001510,-0.006786,0.025848,-0.000583,-0.012092,0.013210,0.026254,0.000084,-0.014601


## Define model and load wights 

In [5]:
# Get embedding columns
emb_cols = [c for c in node_embeddings_df.columns if c.startswith('emb_')]
print(f"Embedding dimensions: {len(emb_cols)}")

# Preview embeddings
node_embeddings_df[['node_id'] + emb_cols[:5]].head()

Embedding dimensions: 32


,node_id,emb_0,emb_1,emb_2,emb_3,emb_4
0,3aa9646b,0.016653,-0.022373,0.029435,0.024201,0.008518
1,1e46e726,0.010128,0.018849,-0.023076,-0.001208,-0.010285
2,49203bc3,-0.002191,-0.026739,0.005370,-0.004439,0.000192
3,a74d1101,-0.003392,-0.017784,-0.009782,-0.026463,-0.020670
4,616d4505,0.022685,-0.000191,-0.003965,-0.002960,-0.015268


## connect hsfs library and get fs handle

In [6]:
# Load alert nodes to join with embeddings
alert_nodes_df = pd.read_csv(os.path.join(TRAINING_DATA_PATH, "alert_nodes_td.csv"))
print(f"Alert nodes: {len(alert_nodes_df)}")
print(f"SAR nodes: {alert_nodes_df['is_sar'].sum()}")

Alert nodes: 7347
SAR nodes: 816


### Get node and edge traininhg dataset objects 

In [7]:
# Create embedding array column (for compatibility with original format)
node_embeddings_df['embedding'] = node_embeddings_df[emb_cols].values.tolist()

# Rename node_id to id for consistency
node_embeddings_df = node_embeddings_df.rename(columns={'node_id': 'id'})

# Select final columns
node_embeddings_final = node_embeddings_df[['id', 'embedding']].copy()
print(f"Final embeddings shape: {node_embeddings_final.shape}")
node_embeddings_final.head()

Final embeddings shape: (7347, 2)


,id,embedding
0,3aa9646b,"[0.01665348, -0.022373362, 0.029434577, 0.0242..."
1,1e46e726,"[0.010128032, 0.018849434, -0.02307619, -0.001..."
2,49203bc3,"[-0.0021910877, -0.026739275, 0.0053704805, -0..."
3,a74d1101,"[-0.0033918873, -0.017783586, -0.009781992, -0..."
4,616d4505,"[0.022684917, -0.00019062453, -0.003965281, -0..."


### Read training datasets as pandas df 

In [8]:
# Join embeddings with alert nodes info
embeddings_with_labels = node_embeddings_df.merge(
    alert_nodes_df[['id', 'is_sar']], 
    on='id', 
    how='left'
)
embeddings_with_labels['is_sar'] = embeddings_with_labels['is_sar'].fillna(0).astype(int)

print(f"Embeddings with labels: {embeddings_with_labels.shape}")
print(f"SAR nodes in embeddings: {embeddings_with_labels['is_sar'].sum()}")

Embeddings with labels: (7347, 35)
SAR nodes in embeddings: 816


### Read hyperparamenter for graph embeddings

In [9]:
# Preview the data
print("Sample of embeddings with SAR labels:")
embeddings_with_labels[['id', 'is_sar'] + emb_cols[:3]].head(10)

Sample of embeddings with SAR labels:


,id,is_sar,emb_0,emb_1,emb_2
0,3aa9646b,0,0.016653,-0.022373,0.029435
1,1e46e726,0,0.010128,0.018849,-0.023076
2,49203bc3,0,-0.002191,-0.026739,0.005370
3,a74d1101,1,-0.003392,-0.017784,-0.009782
4,616d4505,0,0.022685,-0.000191,-0.003965
5,99af2455,1,0.010366,-0.005107,0.029720
6,39be1ea2,0,-0.014126,-0.027495,0.017941
7,e7ec7bdb,1,-0.005520,-0.012380,-0.030405
8,e2e0d938,0,-0.008923,-0.023734,0.011192
9,afc399a9,0,0.023172,0.001367,0.022760


### Construct stellargraph Graph object

In [10]:
# Statistics
print("Embedding statistics:")
print(f"  Total nodes: {len(embeddings_with_labels)}")
print(f"  SAR nodes (is_sar=1): {embeddings_with_labels['is_sar'].sum()}")
print(f"  Non-SAR nodes (is_sar=0): {(embeddings_with_labels['is_sar']==0).sum()}")
print(f"  Embedding dimensions: {len(emb_cols)}")

Embedding statistics:
  Total nodes: 7347
  SAR nodes (is_sar=1): 816
  Non-SAR nodes (is_sar=0): 6531
  Embedding dimensions: 32


### infer node embeddings

In [11]:
# Prepare final feature group data
# Keep id, all embedding columns, and is_sar
final_cols = ['id'] + emb_cols + ['is_sar']
node_embeddings_fg_df = embeddings_with_labels[final_cols].copy()

print(f"Feature group shape: {node_embeddings_fg_df.shape}")
node_embeddings_fg_df.head()

Feature group shape: (7347, 34)


,id,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,emb_7,emb_8,...,emb_23,emb_24,emb_25,emb_26,emb_27,emb_28,emb_29,emb_30,emb_31,is_sar
0,3aa9646b,0.016653,-0.022373,0.029435,0.024201,0.008518,0.019205,0.000119,-0.004495,-0.004817,...,-0.021663,-0.004356,0.005559,-0.011248,0.003737,-0.005930,-0.003396,0.022684,-0.015906,0
1,1e46e726,0.010128,0.018849,-0.023076,-0.001208,-0.010285,-0.014407,0.001506,-0.014259,0.013675,...,-0.017354,-0.014928,0.011635,-0.017038,-0.027755,-0.001501,-0.024140,-0.030260,0.026108,0
2,49203bc3,-0.002191,-0.026739,0.005370,-0.004439,0.000192,0.016502,-0.002795,-0.011158,-0.013902,...,0.012780,-0.029606,-0.030296,0.002708,0.010662,0.014418,-0.027512,0.006895,-0.027158,0
3,a74d1101,-0.003392,-0.017784,-0.009782,-0.026463,-0.020670,0.011362,-0.030823,-0.024861,0.010247,...,0.028176,-0.006826,0.002322,-0.015091,-0.021084,-0.028587,-0.004787,0.010662,-0.022505,1
4,616d4505,0.022685,-0.000191,-0.003965,-0.002960,-0.015268,-0.004893,0.023440,0.018718,-0.019463,...,-0.001510,-0.006786,0.025848,-0.000583,-0.012092,0.013210,0.026254,0.000084,-0.014601,0


In [12]:
# Dummy cell - removed pyspark code

In [13]:
# Dummy cell - removed pyspark code

In [14]:
# Dummy cell - removed pyspark code

In [15]:
# Dummy cell - removed pyspark code

In [16]:
# Dummy cell - removed pyspark code

## Create embeddings feature group

In [17]:
# Save node embeddings feature group locally (replaces hsfs)
fg_path = os.path.join(OUTPUT_PATH, "node_embeddings_fg.parquet")
node_embeddings_fg_df.to_parquet(fg_path, index=False)
print(f"Saved node embeddings feature group to: {fg_path}")

# Also save as CSV for easier inspection
csv_path = os.path.join(OUTPUT_PATH, "node_embeddings_fg.csv")
node_embeddings_fg_df.to_csv(csv_path, index=False)
print(f"Saved CSV version to: {csv_path}")

Saved node embeddings feature group to: /home/adnoman/projects/aml_gan/AMLend2end/output/node_embeddings_fg.parquet


Saved CSV version to: /home/adnoman/projects/aml_gan/AMLend2end/output/node_embeddings_fg.csv


In [18]:
# Summary
print("=" * 50)
print("Node Embeddings Feature Group Created")
print("=" * 50)
print(f"Total nodes: {len(node_embeddings_fg_df)}")
print(f"Embedding dimensions: {len(emb_cols)}")
print(f"SAR nodes: {node_embeddings_fg_df['is_sar'].sum()}")
print(f"Non-SAR nodes: {(node_embeddings_fg_df['is_sar']==0).sum()}")
print(f"\nSaved to: {fg_path}")
print("=" * 50)

Node Embeddings Feature Group Created
Total nodes: 7347
Embedding dimensions: 32
SAR nodes: 816
Non-SAR nodes: 6531

Saved to: /home/adnoman/projects/aml_gan/AMLend2end/output/node_embeddings_fg.parquet


## Feature group provenance
![Feature group provenance](./images/provenance_fg.png)

In [19]:
# Done!